<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_8/Day_3/Daily_challenge%20/Daily_challenge_w8_d3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Build a (RAG) System**
===



Daily Challenge: Build a Retrieval Augmented Generation (RAG) System
===

**What you will create**

You will create a functional RAG system that can answer questions based on a dataset loaded from Hugging Face Datasets. This system will:

Load the databricks/databricks-dolly-15k dataset.
Index the dataset content into a vector store.
Utilize a pre-trained question-answering model from Hugging Face.
Answer user queries by retrieving relevant documents and using the LLM to generate answers.


**1. Set up your environment:**

In [ ]:
!pip install -q langchain torch transformers sentence-transformers datasets faiss-cpu langchain-community langchain-huggingface

import torch
from datasets import load_dataset
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

print("Step 1 Complete: All libraries installed and imported.")

In [ ]:
!pip install -q -U langchain-community langchain-huggingface langchain-classic

**2. Load the dataset:**

In [ ]:
!pip install -Uq datasets

In [ ]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)

data = loader.load()

print(f"Successfully loaded {len(data)} documents.")
print("\n--- Example Entries ---")
print(data[:2])

**3. Split the documents:**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

docs = text_splitter.split_documents(data)

print(f"Original documents: {len(data)}")
print(f"Created {len(docs)} chunks.")
print("\n--- First Chunk ---")
print(docs[0])

**4. Embed the text:**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

modelPath = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

text = "This is a test document."
query_result = embeddings.embed_query(text)

print("Embeddings model loaded.")
print(f"Sample vector (first 3 numbers): {query_result[:3]}")

**5. Create a vector store:**

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

print("Step 5 Complete: Vector store created and indexed.")

**6. Prepare the LLM model:**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id)

gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.7,
    device=-1
)

llm = HuggingFacePipeline(pipeline=gen_pipeline)

print("Step 6 Revised: Generation LLM ready.")

**7. Build the Retrieval QA Chain:**

In [ ]:
from langchain_classic.chains import RetrievalQA

retriever = db.as_retriever(search_kwargs={"k": 3})

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("Step 7 Revised: Chain assembled.")

**8. Test your RAG system:**

In [ ]:
question = "What is cheesemaking?"

result = qa.invoke({"query": question})

print("--- RAG Result ---")
print(f"Question: {question}")
print(f"Answer: {result['result']}")

print("\n--- Sources ---")
for doc in result.get("source_documents", []):
    print(f"- {doc.page_content[:150]}...")